# Phase 1 — Data + Ingestion

This notebook walks through **Phase 1** of the Fact-Checker RAG pipeline:
building the knowledge base every later phase depends on.

RAG needs a knowledge base to retrieve from before it can answer or verify
anything. Phase 1 builds that knowledge base: fetch public data, turn it
into text, chunk it, embed it, store it.

```mermaid
flowchart LR
    WIKI[Wikipedia]
    WB[World Bank]
    FRED[FRED]
    SEC[SEC EDGAR]

    subgraph ingest [" ingest/fetch_*.py — dlt "]
        direction TB
        RAW1[(wikipedia_raw)]
        RAW2[(worldbank_raw)]
        RAW3[(fred_raw)]
        RAW4[(secedgar_raw)]
    end

    WIKI -->|fetch| RAW1
    WB -->|fetch| RAW2
    FRED -->|fetch| RAW3
    SEC -->|fetch| RAW4

    subgraph vectorstore [" ingest/build_vector_store.py "]
        direction TB
        CHUNK[chunk + embed]
        PG[(documents /<br/>document_chunks<br/>pgvector)]
        CHUNK --> PG
    end

    RAW1 --> CHUNK
    RAW2 --> CHUNK
    RAW3 --> CHUNK
    RAW4 --> CHUNK
```

We'll go source by source, then show chunking, embedding, and storage - running real code against the real project, not toy examples.

In [1]:
import sys
sys.path.insert(0, "..")

## The 4 data sources

| Source | What we pull | Why |
|---|---|---|
| **Wikipedia** | 29 articles on financial/economic terms (GDP, Inflation, P/E ratio...) | qualitative context — explains *what a term means* |
| **World Bank** | GDP, inflation, unemployment × 8 countries, yearly | quantitative macro facts, multi-country |
| **FRED** | 5 US macro series, higher frequency (monthly/daily) since 2015 | denser US-only history, complements World Bank |
| **SEC EDGAR** | Revenue / Net income / Total assets × 8 tickers, from 10-K filings | quantitative corporate facts |

Wikipedia gives us prose to explain concepts. The other three give us structured, checkable numbers — exactly what a fact-checker needs to verify a claim against.

### Wikipedia

We don't use the `wikipedia` PyPI package — Wikipedia now requires a `User-Agent` header and that package doesn't send one, so every request gets a `403`. We call the MediaWiki API directly instead.

In [2]:
from ingest.fetch_wikipedia import fetch_wikipedia_article

article = fetch_wikipedia_article("Inflation")
for k, v in article.items():
    print(f"{k}: {v[:300]}")

title: Inflation
url: https://en.wikipedia.org/wiki/Inflation
content: In economics, inflation is an increase in the average price of goods and services in terms of money. This increase is measured using a price index, typically a consumer price index (CPI). When the general price level rises, each unit of currency buys fewer goods and services; consequently, inflation


### World Bank

Structured data: one row per (country, indicator, year). We fetch it with `fetch_worldbank_series(country, code, label)` — a small function that does one HTTP call and shapes the response into rows ready for storage.

In [3]:
from ingest.fetch_worldbank import fetch_worldbank_series

rows = fetch_worldbank_series("US", "NY.GDP.MKTP.CD", "GDP (current US$)")
rows[:3]


[{'country': 'United States',
  'country_code': 'US',
  'indicator_code': 'NY.GDP.MKTP.CD',
  'indicator_label': 'GDP (current US$)',
  'year': 2025,
  'value': 30769700000000.0},
 {'country': 'United States',
  'country_code': 'US',
  'indicator_code': 'NY.GDP.MKTP.CD',
  'indicator_label': 'GDP (current US$)',
  'year': 2024,
  'value': 29298013000000.0},
 {'country': 'United States',
  'country_code': 'US',
  'indicator_code': 'NY.GDP.MKTP.CD',
  'indicator_label': 'GDP (current US$)',
  'year': 2023,
  'value': 27811517000000.0}]

### FRED

Same shape as World Bank: one series per call via `fetch_fred_series(series_id, label)` in `ingest/fetch_fred.py`, using the same `get_json` helper. Needs `FRED_API_KEY` (free, see the README).

In [4]:
from ingest.fetch_fred import fetch_fred_series

rows = fetch_fred_series("GDP", "Gross Domestic Product")
rows[-3:]

[{'series_id': 'GDP',
  'series_label': 'Gross Domestic Product',
  'date': '2025-07-01',
  'value': 31098.027},
 {'series_id': 'GDP',
  'series_label': 'Gross Domestic Product',
  'date': '2025-10-01',
  'value': 31422.526},
 {'series_id': 'GDP',
  'series_label': 'Gross Domestic Product',
  'date': '2026-01-01',
  'value': 31865.721}]

### SEC EDGAR

The messiest of the four. Two lessons learned here, both about data that looks fine until you check it closely:

- Companies **change XBRL tags for revenue over time**. The ASC 606 accounting transition moved the tag from `Revenues` to `RevenueFromContractWithCustomerExcludingAssessedTax`. Taking only the first tag with data loses history — Apple came back with 3 years instead of 10. Fix: merge both tags, dedupe by period-end date, keep the most recently filed value (`annual_values()` in `ingest/fetch_secedgar.py`).
- The SEC also **requires a real `User-Agent`** with contact info, same as Wikipedia — anonymous requests get rejected.

In [7]:
from ingest.fetch_secedgar import USER_AGENT, fetch_company_facts
from ingest.http import get_json

ticker_map = get_json(
    "https://www.sec.gov/files/company_tickers.json",
    headers={"User-Agent": USER_AGENT},
    timeout=20,
)
rows = fetch_company_facts("AAPL", ticker_map)
rows[:3]

[{'ticker': 'AAPL',
  'company_name': 'Apple Inc.',
  'cik': '0000320193',
  'tag': 'Revenue',
  'fiscal_year_end': '2016-09-24',
  'value_usd': 215639000000,
  'form': '10-K',
  'filed': '2018-11-05'},
 {'ticker': 'AAPL',
  'company_name': 'Apple Inc.',
  'cik': '0000320193',
  'tag': 'Revenue',
  'fiscal_year_end': '2017-09-30',
  'value_usd': 229234000000,
  'form': '10-K',
  'filed': '2019-10-31'},
 {'ticker': 'AAPL',
  'company_name': 'Apple Inc.',
  'cik': '0000320193',
  'tag': 'Revenue',
  'fiscal_year_end': '2018-09-29',
  'value_usd': 265595000000,
  'form': '10-K',
  'filed': '2020-10-30'}]

## Ingestion: dlt

Each source above is wrapped in a small [dlt](https://dlthub.com/) resource in `ingest/fetch_*.py`. dlt loads each source into its **own raw
staging schema** in Postgres, with `write_disposition="merge"` — running a fetch twice doesn't duplicate rows, dlt merges by primary key.

```python
@dlt.resource(
    name="worldbank_indicators",
    write_disposition="merge",
    primary_key=["country_code", "indicator_code", "year"]
    )
def worldbank_indicators():
    for country in WB_COUNTRIES:
        for code, label in WB_INDICATORS.items():
            yield from fetch_worldbank_series(country, code, label)
```

**Lesson learned:** dlt silently hides type mismatches. If a resource yields a field as `int` in some rows and `float` in others (World Bank's
GDP came back as `int`, inflation as `float`), dlt creates a *shadow variant column* (`value__v_double`) and leaves the original `value` column `NULL` for the mismatched rows. 1156 of 1226 rows were silently `NULL` before we caught it — via `count(*) FILTER (WHERE value IS NULL)`. Fix: always cast numeric fields to the same type (`float(...)`) before `yield`.

We already ran all 4 fetchers for this project (see Quick Start in the README) — this notebook reads from that already-populated database rather than re-running long fetches.

## Chunking

Once raw data is in Postgres, `ingest/build_vector_store.py` turns rows into chunks. Two different strategies, by data shape:

- **Wikipedia** (long prose) → sentence-level chunking, 4 sentences per chunk with 1-sentence overlap (`src/chunking.py`).
- **World Bank / FRED / SEC EDGAR** (structured facts) → no chunking needed. Each row already *is* one fact — we template it into a sentence instead, e.g. `"GDP for United States in 2025 was 30769700000000.00."` One fact, one chunk.

The sentence splitter is a plain regex, no NLTK model download needed.
That's fine for most text — but it has a sharp edge with citations:

In [8]:
from src.chunking import split_sentences

citation = (
    'Clifford Cobb, Ted Halstead and Jonathan Rowe. "If the GDP is up, '
    'why is America down?" The Atlantic Monthly, vol. 276, no. 4, '
    "October 1995, pages 59-78"
)
split_sentences(citation)


['Clifford Cobb, Ted Halstead and Jonathan Rowe. "If the GDP is up, why is America down?" The Atlantic Monthly, vol. 276, no. 4, October 1995, pages 59-78']

That comes back as **one** sentence, not three. Without an abbreviation whitelist, a naive `[.!?]\s+[A-Z0-9]` regex would split after `vol.` and `no.` too — a period followed by a space and a digit looks exactly like a sentence boundary. `vol. 276, no. 4` would get torn into 3 fragments, scattering a citation's volume/issue number away from the fact it belongs to. `src/chunking.py` keeps a small whitelist (`vol`, `no`, `pp`, `e.g`, `i.e`, ...) and checks the word before the punctuation before deciding to split.

## Embedding

Every chunk — sentence or templated fact — goes through the same
embedding call:

In [9]:
from src.embeddings import embed_texts

vectors = embed_texts(["Apple's revenue for fiscal year 2025 was $416B."])
print("dimensions:", len(vectors[0]))
print("first 5 values:", vectors[0][:5])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

dimensions: 384
first 5 values: [0.050689924508333206, 0.050583891570568085, 0.10011453926563263, -0.011838517151772976, -0.052752889692783356]


`sentence-transformers/all-MiniLM-L6-v2`, 384 dimensions, runs locally —
no API cost, no network call, good enough for MVP-scale retrieval. This
is the same model used for every source; structured facts and prose
articles land in the same 384-dim vector space so they're searchable
together.

## Storage: Postgres + pgvector

One database holds both the raw staging schemas (one per source) and the
shared vector store (`documents` + `document_chunks`). No separate
vector DB.

```sql
CREATE TABLE document_chunks (
    id UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    document_id UUID REFERENCES documents(id) ON DELETE CASCADE,
    content TEXT NOT NULL,
    embedding VECTOR(384),
    metadata JSONB DEFAULT '{}',
    chunk_index INTEGER,
    content_tsv tsvector GENERATED ALWAYS AS (to_tsvector('english', content)) STORED
);

CREATE INDEX idx_chunks_embedding ON document_chunks USING hnsw (embedding vector_cosine_ops);
CREATE INDEX idx_chunks_fts ON document_chunks USING GIN (content_tsv);
```

The HNSW index (approximate vector search) and the generated `tsvector` column (full-text search) are both already in place — set up now so Phase 3's hybrid search has an index to query, even though nothing queries it yet.

Let's check what's actually in there right now:

In [10]:
from src.db import get_conn

with get_conn() as conn, conn.cursor() as cur:
    cur.execute("SELECT source, count(*) FROM documents GROUP BY source ORDER BY source")
    print("documents by source:", cur.fetchall())
    cur.execute("SELECT count(*) FROM document_chunks")
    print("total chunks:", cur.fetchone()[0])


documents by source: [('fred', 5), ('secedgar', 24), ('wikipedia', 29), ('worldbank', 24)]
total chunks: 6846


In [11]:
with get_conn() as conn, conn.cursor() as cur:
    cur.execute("""
        SELECT content, metadata
        FROM document_chunks
        WHERE metadata->>'source' = 'secedgar'
        LIMIT 3
    """)
    for content, metadata in cur.fetchall():
        print(metadata)
        print(" ", content)
        print()


{'tag': 'Net income', 'source': 'secedgar', 'ticker': 'AAPL'}
  Apple Inc. (AAPL) reported Net income of $59,531,000,000 for fiscal year ending 2018-09-29 (10-K filed 2020-10-30).

{'tag': 'Net income', 'source': 'secedgar', 'ticker': 'AAPL'}
  Apple Inc. (AAPL) reported Net income of $55,256,000,000 for fiscal year ending 2019-09-28 (10-K filed 2021-10-29).

{'tag': 'Net income', 'source': 'secedgar', 'ticker': 'AAPL'}
  Apple Inc. (AAPL) reported Net income of $57,411,000,000 for fiscal year ending 2020-09-26 (10-K filed 2022-10-28).



## Running the full pipeline yourself

This notebook only made a few small live calls. The full ingestion — what actually populated the database above — is 5 commands (see the README Quick Start):

```bash
docker compose up -d postgres
uv run python -m ingest.fetch_wikipedia
uv run python -m ingest.fetch_worldbank
uv run python -m ingest.fetch_fred        # needs FRED_API_KEY
uv run python -m ingest.fetch_secedgar
uv run python -m ingest.build_vector_store
```

The first 4 fetch raw data into their own schemas (safe to re-run — dlt merges by primary key, no duplicates). The last one rebuilds `documents`/`document_chunks` from scratch every time: it truncates both tables before re-chunking and re-embedding everything.

That truncate exists because of a real bug: without it, running `build_vector_store.py` twice **silently doubled every document** — 29 Wikipedia articles became 58, 24 World Bank series became 48. No error, no warning, just twice the rows. `documents.title` has no unique constraint, so nothing stopped it. The fix is a one-line `TRUNCATE` before the rebuild — acceptable for now since Phase 1 doesn't run on a schedule yet. It'll need to become an upsert once Phase 2's Airflow DAG starts running this daily.

## Recap

- 4 sources, each a small dlt resource → own raw Postgres schema.
- Wikipedia gets sentence-chunked; World Bank/FRED/SEC EDGAR get templated into one-fact-one-chunk sentences.
- Everything embeds through the same local model into the same 384-dim space.
- One Postgres database holds raw + vector store, with HNSW and full-text indexes already built for later.
- Real bugs caught along the way: dlt's silent type-variant columns, SEC EDGAR's XBRL tag drift, a naive sentence-splitter mis-handling citations, and a missing truncate that silently duplicated the whole vector store.

Phase 1 is done: a populated, reproducible knowledge base. Phase 2 builds the claim extractor and RAG chain on top of it — not covered here.